# Gemma 4 Research Agent — Phase 2: LangChain

Phase 2 refactors the raw Python agent from Phase 1 into LangChain components, replacing each piece one-to-one.

**Phase 1 → Phase 2 mapping:**

| Phase 1 (raw Python) | Phase 2 (LangChain) |
|---|---|
| `generate_response()` | `ChatHuggingFace.invoke()` |
| `build_planner_user_prompt()` | `ChatPromptTemplate` (PLANNER_PROMPT) |
| `build_synthesizer_user_prompt()` | `ChatPromptTemplate` (SYNTHESIZER_PROMPT) |
| `_strip_thinking_tokens()` | `clean_response()` via `RunnableLambda` |
| `search_web()` | `DuckDuckGoSearchRun.invoke()` |
| `format_search_results_for_prompt()` | `run_searches()` returns concatenated string |
| `run_research_agent()` loop | LCEL pipeline: `prompt \| chat_model \| StrOutputParser() \| RunnableLambda` |

**Stack:**
| Component | Tool |
|---|---|
| Model | `google/gemma-4-12B-it` via HuggingFace Transformers |
| LangChain wrapper | `langchain-huggingface` — `HuggingFacePipeline` + `ChatHuggingFace` |
| Web search | `DuckDuckGoSearchRun` via `langchain-community` |
| Tracing | LangSmith (`langchain-core` tracing v2) |
| Runtime | Kaggle T4 x2 GPU |

**References:**
- LCEL: https://python.langchain.com/docs/concepts/lcel/
- ChatHuggingFace: https://python.langchain.com/docs/integrations/chat/huggingface/
- LangSmith: https://docs.smith.langchain.com/
- Gemma 4 model card: https://huggingface.co/google/gemma-4-12B-it
- Phase 1 notebook: ../Research_Agent_Gemma4/notebook.ipynb

## Prerequisites

Before running this notebook:

1. **Accept the Gemma 4 license** on HuggingFace: https://huggingface.co/google/gemma-4-12B-it
2. **Create a HuggingFace token** (Read access): https://huggingface.co/settings/tokens
3. **Create a LangSmith account and API key**: https://smith.langchain.com/settings
4. **Add both to Kaggle Secrets** — names: `HF_TOKEN` and `LANGCHAIN_API_KEY`, Notebook access toggled on
5. **Set accelerator to GPU T4 x2** in notebook Settings
6. **Enable Internet** in notebook Settings

## Step 1 — Install Dependencies

In [ ]:
!pip install -q -U transformers langchain langchain-huggingface langchain-community langsmith ddgs accelerate bitsandbytes

> **Restart the kernel after installation** before proceeding.
> Run → Restart & Clear Outputs, then continue from Step 2.

## Step 2 — Authentication

In [ ]:
import os
import warnings
from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore", message="Both `max_new_tokens`")

user_secrets = UserSecretsClient()

os.environ["HF_TOKEN"]               = user_secrets.get_secret("HF_TOKEN")
os.environ["LANGCHAIN_API_KEY"]      = user_secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]   = "true"
os.environ["LANGCHAIN_PROJECT"]      = "gemma4-research-agent-phase2"

print("HF_TOKEN set:", os.environ["HF_TOKEN"][:8] + "...")
print("LangSmith tracing enabled.")
print("Project:", os.environ["LANGCHAIN_PROJECT"])

## Step 3 — Load Model

Loads `google/gemma-4-12B-it` in 4-bit NF4 quantization and wraps it in `ChatHuggingFace`.

**Why `ChatHuggingFace` and not `HuggingFacePipeline` directly:**
`HuggingFacePipeline.invoke()` passes raw strings to the model, bypassing `apply_chat_template`. Instruction-tuned models like Gemma 4 require structured chat formatting — without it the model produces token repetition instead of a coherent response. `ChatHuggingFace` wraps the pipeline and applies the correct chat template automatically.

References:
- `AutoModelForImageTextToText`: https://huggingface.co/docs/transformers/main/en/model_doc/gemma4_unified
- `ChatHuggingFace`: https://python.langchain.com/docs/integrations/chat/huggingface/

In [ ]:
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_ID = "google/gemma-4-12B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()
model.generation_config.max_length     = None
model.generation_config.max_new_tokens = 2048
model.generation_config.temperature    = 0.7
model.generation_config.top_p          = 0.95
model.generation_config.do_sample      = True

print("Memory footprint:", round(model.get_memory_footprint() / 1e9, 2), "GB")
print("Device map:", model.hf_device_map)

In [ ]:
generation_pipeline = pipeline(
    task="image-text-to-text",
    model=model,
    processor=processor
)

llm        = HuggingFacePipeline(pipeline=generation_pipeline)
chat_model = ChatHuggingFace(llm=llm)

print("ChatHuggingFace ready:", type(chat_model))

## Step 4 — Load Agent Modules

Defines all functions from `model.py`, `tools.py`, `prompts.py`, and `agent.py`.

See the project repo for individual source files:
https://github.com/dwinsi/LLMfromScratch/tree/main/Research_Agent_Gemma4/phase2_langchain

In [ ]:
# ── model.py ────────────────────────────────────────────────────────────────

def clean_response(raw: str) -> str:
    """Strips Gemma 4 special tokens from generated output."""
    raw = re.sub(r"<\|channel>thought.*?<channel\|>", "", raw, flags=re.DOTALL)
    if "<|turn>model" in raw:
        raw = raw.split("<|turn>model")[-1]
    raw = re.sub(r"<\|turn>|<turn\|>|<bos>|<eos>", "", raw)
    return raw.strip()


# ── tools.py ────────────────────────────────────────────────────────────────

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper     = DuckDuckGoSearchAPIWrapper(max_results=3)
search_tool = DuckDuckGoSearchRun(api_wrapper=wrapper)


def run_searches(sub_questions: list[str]) -> str:
    all_results = []
    for index, question in enumerate(sub_questions, start=1):
        print(f"[{index}/{len(sub_questions)}] Searching: {question}")
        try:
            result = search_tool.invoke(question)
            if result:
                all_results.append(f'Search results for: "{question}"\n{result}')
                print(f"  Done.\n")
        except Exception as error:
            print(f"  Failed: {error}. Skipping.\n")
    return "\n\n".join(all_results)


# ── prompts.py ───────────────────────────────────────────────────────────────

from langchain_core.prompts import ChatPromptTemplate

PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a research planner. Your only job is to break a complex question into 2 or 3 focused sub-questions that can each be answered by a single web search.

Rules you must follow without exception:
- Output ONLY a numbered list. No introduction, no explanation, no closing remarks.
- Each sub-question must be specific and self-contained.
- Each sub-question must be on its own line.
- Do not number beyond 3.
- Do not output anything other than the numbered list.

Example output format:
1. What is the history of quantum computing?
2. What are the latest breakthroughs in quantum computing in 2025?
3. Which companies are leading quantum computing research today?"""),
    ("human", "Break this research question into 2 or 3 focused sub-questions:\n\n{question}")
])

SYNTHESIZER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a research synthesizer. You receive a research question and a set of web search results. Your job is to write a clear, well-structured answer based strictly on the search results provided.

Rules you must follow:
- Base your answer only on the search results provided. Do not add information from outside the results.
- Structure your answer with a short introduction, key findings, and a brief conclusion.
- Write in plain, precise language. No filler phrases."""),
    ("human", "Research question: {question}\n\nSearch results:\n{search_results}\n\nWrite a structured answer based on the search results above.")
])

def parse_sub_questions(planner_output: str) -> list[str]:
    sub_questions = []
    for line in planner_output.strip().splitlines():
        cleaned = line.strip()
        if not cleaned:
            continue
        if cleaned[0].isdigit():
            stripped = cleaned.lstrip("0123456789").lstrip(".").lstrip(")").strip()
            if stripped:
                sub_questions.append(stripped)
    return sub_questions


# ── agent.py ─────────────────────────────────────────────────────────────────

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.tracers.context import tracing_v2_enabled

def _print_step_header(title: str) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}\n")

def run_research_agent_lcel(user_question: str) -> str:
    with tracing_v2_enabled(project_name=os.environ.get("LANGCHAIN_PROJECT", "gemma4-research-agent-phase2")):

        _print_step_header("RESEARCH AGENT STARTING (Phase 2 — LCEL)")
        print(f"Question: {user_question}\n")

        # Step 1 — Planner
        _print_step_header("STEP 1: PLANNING")
        planner_chain = PLANNER_PROMPT | chat_model | StrOutputParser() | RunnableLambda(clean_response)
        planner_output = planner_chain.invoke({"question": user_question})
        print(f"Planner output:\n{planner_output}\n")

        sub_questions = parse_sub_questions(planner_output)
        if not sub_questions:
            return "Agent failed: planner did not produce valid sub-questions."

        print(f"Parsed {len(sub_questions)} sub-questions:")
        for index, question in enumerate(sub_questions, start=1):
            print(f"  {index}. {question}")

        # Step 2 — Search
        _print_step_header("STEP 2: SEARCHING")
        search_results = run_searches(sub_questions)
        if not search_results:
            return "Agent failed: no search results returned."

        # Step 3 — Synthesizer
        _print_step_header("STEP 3: SYNTHESIZING")
        synthesizer_chain = SYNTHESIZER_PROMPT | chat_model | StrOutputParser() | RunnableLambda(clean_response)
        final_answer = synthesizer_chain.invoke({
            "question": user_question,
            "search_results": search_results
        })

        _print_step_header("FINAL ANSWER")
        print(final_answer)

        return final_answer


print("All agent modules loaded.")

## Step 5 — Run the Agent

Change `user_question` to any research topic and run.

Every run is traced to LangSmith under the project `gemma4-research-agent-phase2`:
https://smith.langchain.com

In [ ]:
user_question = "How do transformer models handle long-range dependencies?"

answer = run_research_agent_lcel(user_question)

## What LangChain Added

| Capability | Phase 1 | Phase 2 |
|---|---|---|
| Chat formatting | Manual `apply_chat_template()` | `ChatHuggingFace` handles automatically |
| Prompt management | Plain string functions | `ChatPromptTemplate` with named variables |
| Output parsing | Custom `parse_sub_questions()` | `StrOutputParser` + `RunnableLambda` |
| Post-processing | Called manually after generate | Sits in the pipe chain |
| Observability | `print()` statements | LangSmith traces every step |
| Chaining | Explicit function calls | `prompt \| model \| parser \| lambda` |

## What Phase 1 Still Does Better

| Capability | Phase 1 | Phase 2 |
|---|---|---|
| Search result structure | Structured list with title, url, snippet | Single concatenated string |
| Citation numbers | `[1]`, `[2]` per source | Not available |
| Debugging visibility | Every variable inspectable | Abstracted inside chain |

## What's Next

Phase 3 — swap `ChatHuggingFace` for a hosted model (`ChatGoogleGenerativeAI` or `ChatOpenAI`) and replace the fixed LCEL pipeline with `AgentExecutor` + ReAct for a fully dynamic tool-calling loop.